### 01 Data Understanding

#### Project Context
#####
This project builds a healthcare data warehouse using SQL, Python, Tableau, Streamlit, and GitHub.

The goal is to transform raw hospital patient records into a structured analytics-ready star schema that supports reporting on:

- Patient demographics
- Hospital admissions
- Medical conditions
- Billing amounts
- Insurance providers
- Doctors
- Medications
- Length of stay

Load the CSV

In [1]:
import pandas as pd
import os

DATA_DIR = "../data/raw"

files = os.listdir(DATA_DIR)
files

['hospital data analysis.csv']

In [2]:
file_path = os.path.join(DATA_DIR, files[0])

df = pd.read_csv(file_path)

df.head()

,Patient_ID,Age,Gender,Condition,Procedure,Cost,Length_of_Stay,Readmission,Outcome,Satisfaction
0,1,45,Female,Heart Disease,Angioplasty,15000,5,No,Recovered,4
1,2,60,Male,Diabetes,Insulin Therapy,2000,3,Yes,Stable,3
2,3,32,Female,Fractured Arm,X-Ray and Splint,500,1,No,Recovered,5
3,4,75,Male,Stroke,CT Scan and Medication,10000,7,Yes,Stable,2
4,5,50,Female,Cancer,Surgery and Chemotherapy,25000,10,No,Recovered,4


In [3]:
df.shape

(984, 10)

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 984 entries, 0 to 983
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   Patient_ID      984 non-null    int64
 1   Age             984 non-null    int64
 2   Gender          984 non-null    str  
 3   Condition       984 non-null    str  
 4   Procedure       984 non-null    str  
 5   Cost            984 non-null    int64
 6   Length_of_Stay  984 non-null    int64
 7   Readmission     984 non-null    str  
 8   Outcome         984 non-null    str  
 9   Satisfaction    984 non-null    int64
dtypes: int64(5), str(5)
memory usage: 122.9 KB


In [5]:
df.columns

Index(['Patient_ID', 'Age', 'Gender', 'Condition', 'Procedure', 'Cost',
       'Length_of_Stay', 'Readmission', 'Outcome', 'Satisfaction'],
      dtype='str')

In [6]:
df.isnull().sum()

Patient_ID        0
Age               0
Gender            0
Condition         0
Procedure         0
Cost              0
Length_of_Stay    0
Readmission       0
Outcome           0
Satisfaction      0
dtype: int64

In [7]:
df.duplicated().sum()

np.int64(0)

In [8]:
df.describe(include="all")

,Patient_ID,Age,Gender,Condition,Procedure,Cost,Length_of_Stay,Readmission,Outcome,Satisfaction
count,984.000000,984.000000,984,984,984,984.000000,984.000000,984,984,984.000000
unique,NaN,NaN,2,15,15,NaN,NaN,2,2,NaN
top,NaN,NaN,Female,Fractured Leg,Cast and Physical Therapy,NaN,NaN,No,Recovered,NaN
freq,NaN,NaN,524,67,67,NaN,NaN,720,591,NaN
mean,500.329268,53.754065,NaN,NaN,NaN,8367.479675,37.663618,NaN,NaN,3.598577
std,288.979531,14.941135,NaN,NaN,NaN,7761.990976,19.595805,NaN,NaN,0.883002
min,1.000000,25.000000,NaN,NaN,NaN,100.000000,1.000000,NaN,NaN,2.000000
25%,250.750000,45.000000,NaN,NaN,NaN,1000.000000,21.000000,NaN,NaN,3.000000
50%,500.500000,55.000000,NaN,NaN,NaN,6000.000000,38.000000,NaN,NaN,4.000000
75%,750.250000,65.000000,NaN,NaN,NaN,15000.000000,54.250000,NaN,NaN,4.000000


### Business Entity Identification

Based on the raw hospital dataset, the following business entities are identified:

#### Patient Entity
Represents people receiving healthcare services.

Potential columns:
- Name
- Age
- Gender
- Blood Type

#### Hospital Entity
Represents healthcare facilities where patients are admitted.

Potential columns:
- Hospital

#### Doctor Entity
Represents healthcare providers responsible for patient care.

Potential columns:
- Doctor

#### Insurance Entity
Represents the payer or insurance provider.

Potential columns:
- Insurance Provider

#### Medical Condition Entity
Represents diagnosed conditions.

Potential columns:
- Medical Condition

#### Medication Entity
Represents medications prescribed or administered.

Potential columns:
- Medication

#### Visit / Admission Fact Entity
Represents the measurable healthcare event.

Potential measures:
- Billing Amount
- Length of Stay
- Admission Count

In [9]:
df.columns

Index(['Patient_ID', 'Age', 'Gender', 'Condition', 'Procedure', 'Cost',
       'Length_of_Stay', 'Readmission', 'Outcome', 'Satisfaction'],
      dtype='str')

In [10]:
for col in df.columns:
    print(col)

Patient_ID
Age
Gender
Condition
Procedure
Cost
Length_of_Stay
Readmission
Outcome
Satisfaction


Star Schema for Your Dataset
Fact Table

fact_patient_outcome

Measures

- Cost
- Length_of_Stay
- Satisfaction

Dimensions Tables 
- dim_patient
- dim_condition
- dim_procedure
- dim_outcome

Analyze Business Entities

In [11]:
# Patients
df["Patient_ID"].nunique()

984

In [12]:
# Conditions
df["Condition"].value_counts()

Condition
Fractured Leg            67
Heart Attack             67
Fractured Arm            66
Stroke                   66
Cancer                   66
Hypertension             66
Appendicitis             66
Allergic Reaction        66
Heart Disease            65
Diabetes                 65
Respiratory Infection    65
Prostate Cancer          65
Childbirth               65
Kidney Stones            65
Osteoarthritis           64
Name: count, dtype: int64

In [13]:
# Procedures
df["Procedure"].value_counts()

Procedure
Cast and Physical Therapy               67
Cardiac Catheterization                 67
X-Ray and Splint                        66
CT Scan and Medication                  66
Surgery and Chemotherapy                66
Medication and Counseling               66
Appendectomy                            66
Epinephrine Injection                   66
Angioplasty                             65
Insulin Therapy                         65
Antibiotics and Rest                    65
Radiation Therapy                       65
Delivery and Postnatal Care             65
Lithotripsy                             65
Physical Therapy and Pain Management    64
Name: count, dtype: int64

In [14]:
# Outcomes

df["Outcome"].value_counts()

Outcome
Recovered    591
Stable       393
Name: count, dtype: int64

In [15]:
# Readmissions

df["Readmission"].value_counts()

Readmission
No     720
Yes    264
Name: count, dtype: int64

In [16]:
# Cost Analysis

df["Cost"].describe()

count      984.000000
mean      8367.479675
std       7761.990976
min        100.000000
25%       1000.000000
50%       6000.000000
75%      15000.000000
max      25000.000000
Name: Cost, dtype: float64

In [17]:
# Length of Stay Anaysis

df["Length_of_Stay"].describe()

count    984.000000
mean      37.663618
std       19.595805
min        1.000000
25%       21.000000
50%       38.000000
75%       54.250000
max       76.000000
Name: Length_of_Stay, dtype: float64

In [18]:
# Satisfaction Analysis

df["Satisfaction"].describe()

count    984.000000
mean       3.598577
std        0.883002
min        2.000000
25%        3.000000
50%        4.000000
75%        4.000000
max        5.000000
Name: Satisfaction, dtype: float64

In [19]:
# Save Processed Dataset

os.makedirs("../data/processed", exist_ok=True)

In [20]:
df.to_csv(
    "../data/processed/healthcare_processed.csv",
    index=False
)

In [21]:
print(df["Condition"].nunique())
print(df["Procedure"].nunique())
print(df["Outcome"].nunique())

15
15
2
